# 01 – Exploration : charge d'entraînement et récupération

Questions :
1. Comment ma charge se répartit-elle entre les sports, semaine par semaine ?
2. Comment évoluent ma VFC et ma fréquence cardiaque de repos ?
3. Ma charge (du jour ou accumulée) a-t-elle un effet sur ma récupération des nuits suivantes ?
4. L'heure des séances compte-t-elle ?

⚠️ Dépôt public : les sorties de ce notebook contiennent des données personnelles. Elles sont effacées automatiquement au commit (nbstripout).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GOLD = Path("..") / "data" / "gold"
SILVER = Path("..") / "data" / "silver"

d = pd.read_parquet(GOLD / "daily_features.parquet")
act = pd.read_parquet(SILVER / "activities.parquet")
d["date"] = pd.to_datetime(d["date"])
act["start_time"] = pd.to_datetime(act["start_time"])

# Écart de VFC à sa moyenne glissante sur 7 jours (neutralise l'état de départ)
d["hrv_dev"] = d["hrv_last_night"] - d["hrv_last_night"].rolling(7, min_periods=4).mean()

plt.rcParams["figure.figsize"] = (12, 4)
print(f"{len(d)} jours, {len(act)} activités, {d['night_tracked'].sum()} nuits suivies")

## 1. Charge hebdomadaire par sport (TRIMP)

In [ ]:
sport_cols = [c for c in d.columns if c.startswith("load_") and c != "load_total"]
weekly = d.set_index("date")[sport_cols].resample("W").sum()
weekly.columns = [c.removeprefix("load_") for c in weekly.columns]

ax = weekly.plot(kind="bar", stacked=True, width=0.85)
ax.set_xticklabels([w.strftime("%d/%m") for w in weekly.index], rotation=45)
ax.set_ylabel("TRIMP")
ax.set_title("Charge hebdomadaire par sport")
plt.tight_layout()

## 2. Récupération : VFC et FC de repos (moyenne glissante 7 jours)

In [ ]:
fig, ax1 = plt.subplots()
s = d.set_index("date")
ax1.plot(s.index, s["hrv_last_night"], ".", alpha=0.3, color="tab:blue")
ax1.plot(s.index, s["hrv_last_night"].rolling(7, min_periods=4).mean(), color="tab:blue", label="VFC (ms)")
ax1.set_ylabel("VFC (ms)", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(s.index, s["resting_hr"].rolling(7, min_periods=4).mean(), color="tab:red", label="FC repos (bpm)")
ax2.set_ylabel("FC repos (bpm)", color="tab:red")
ax1.set_title("VFC et FC de repos")
plt.tight_layout()

## 3. Forme et fatigue : ATL, CTL, TSB

- **ATL** (charge aiguë, ~7 j) : fatigue récente
- **CTL** (charge chronique, ~42 j) : forme de fond
- **TSB** = CTL − ATL : fraîcheur (négatif = plus fatigué que d'habitude)

In [ ]:
ax = s[["atl", "ctl"]].plot()
ax.fill_between(s.index, s["tsb"], 0, where=s["tsb"] < 0, color="tab:red", alpha=0.2, label="TSB < 0")
ax.fill_between(s.index, s["tsb"], 0, where=s["tsb"] >= 0, color="tab:green", alpha=0.2, label="TSB ≥ 0")
ax.legend()
ax.set_title("Charge aiguë, chronique et fraîcheur")
plt.tight_layout()

## 4. Effet de la charge sur la récupération, avec décalage

Corrélation entre la charge du jour J et l'écart de VFC à J+k. La zone grise correspond au bruit (±2/√n) : une barre qui reste dedans n'est pas distinguable du hasard.

In [ ]:
lags = range(1, 6)
rows = []
for k in lags:
    target = d["hrv_dev"].shift(-k)
    rows.append({
        "décalage": f"J+{k}",
        "charge du jour": d["load_total"].corr(target),
        "charge accumulée (ATL)": d["atl"].corr(target),
    })
corr = pd.DataFrame(rows).set_index("décalage")

n = d["hrv_dev"].notna().sum()
seuil = 2 / np.sqrt(n)
ax = corr.plot(kind="bar", rot=0)
ax.axhspan(-seuil, seuil, color="grey", alpha=0.2)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("corrélation")
ax.set_title(f"Charge → écart de VFC (n = {n}, seuil ≈ ±{seuil:.2f})")
plt.tight_layout()
corr.round(2)

## 5. L'heure des séances compte-t-elle ?

Hypothèse : une séance tardive (match de tennis le soir) perturbe davantage la nuit suivante qu'une séance plus tôt.

In [ ]:
act["hour"] = act["start_time"].dt.hour
act["date"] = act["start_time"].dt.normalize()
last_session = act.groupby("date")["hour"].max().rename("last_hour")

e = d.set_index("date").join(last_session)
e["hrv_dev_next"] = e["hrv_dev"].shift(-1)
e["sleep_h_next"] = e["sleep_h"].shift(-1)
e["moment"] = pd.cut(e["last_hour"], bins=[-1, 17, 19, 24], labels=["avant 18 h", "18 h – 20 h", "après 20 h"])
e["moment"] = e["moment"].cat.add_categories("repos").fillna("repos")

summary = e.groupby("moment", observed=False)[["hrv_dev_next", "sleep_h_next"]].agg(["count", "mean"]).round(2)

fig, axes = plt.subplots(1, 2)
groups = ["repos", "avant 18 h", "18 h – 20 h", "après 20 h"]
for ax, col, title in zip(axes, ["hrv_dev_next", "sleep_h_next"], ["Écart de VFC la nuit suivante", "Sommeil la nuit suivante (h)"]):
    data = [e.loc[e["moment"] == g, col].dropna() for g in groups]
    ax.boxplot(data, tick_labels=groups)
    ax.set_title(title)
plt.tight_layout()
summary

## Conclusions

**Répartition de la charge.** Le tennis est la principale source de charge, devant la course, avec une charge par séance comparable (~110 TRIMP). La musculation apparaît très faible (~23 TRIMP) : c'est une limite connue du TRIMP, fondé sur la fréquence cardiaque, qui sous-estime l'effort musculaire.

**Effet de la charge sur la récupération.** Aucune corrélation n'est significative (n = 146, seuil ≈ ±0,17). La charge du jour n'a pas d'effet visible sur la VFC des nuits suivantes. La charge accumulée (ATL) montre en revanche une tendance négative cohérente de J+2 à J+5 (jusqu'à −0,11), dans le sens attendu d'une fatigue cumulée, mais elle reste dans le bruit. À réévaluer quand les données se seront accumulées.

**Effet de l'heure des séances.** Contrairement à l'hypothèse, les séances du soir ne dégradent pas la nuit suivante : la VFC y est au-dessus de la moyenne (+9 à +10 ms), et les nuits suivant un entraînement sont plus longues (7,2 à 7,7 h) que les nuits de repos (6,7 h). Ces résultats sont fragiles : les groupes du soir sont petits (17 et 7 jours), et les jours de repos pourraient être confondus avec le jour de la semaine (week-ends, couchers tardifs).

**Qualité des données.** Les jours de repos concentrent des nuits très courtes (< 3 h) et des écarts de VFC extrêmes, qui pourraient correspondre à des nuits partiellement enregistrées plutôt qu'à de vraies nuits courtes.

**Conséquences pour le modèle de récupération :**
- ajouter le jour de la semaine, le fait d'être un jour de repos, l'heure de la dernière séance et l'ATL comme variables ;
- signaler les nuits de moins de 4 h comme suspectes et les exclure de l'entraînement ;
- comparer systématiquement le modèle à une référence simple (VFC de demain = VFC d'aujourd'hui) : il ne sera utile que s'il fait mieux.